# Symmetric Cryptography High Level Overview

In [1]:
import hmac
import hashlib

# Symmetric Cryptography

*One shared key*

### Setting

<div style="text-align:center">
    <img src="./img/AB/014_AB.svg" style="vertical-align:middle;width:100%">
</div>

### Setting

<div style="text-align:center">
    <img src="./img/AB/015_AB.svg" style="vertical-align:middle;width:100%">
</div>

### Setting

<div style="text-align:center">
    <img src="./img/AB/016_AB.svg" style="vertical-align:middle;width:100%">
</div> 

### Setting

<div style="text-align:center">
    <img src="./img/AB/017_AB.svg" style="vertical-align:middle;width:100%">
</div>    

## Terminology

* **Alice** sends a message to **Bob** over a channel she does not control
* **Eve** only *listens* (passive) &middot; **Mallory** *modifies (active)
* Encryption terminology:
    + $m$ &nbsp;Message or plaintext 
    + $c$ &nbsp;Ciphertext 
    + $k$ &nbsp;Key (must be kept secret)
    + `Enc(k, m) = c` &nbsp;&rarr;&nbsp; `Dec(k, c) = m` 

<div style="flex:4; display:flex; flex-direction:column; gap:0.5em">
    <img src="./img/AB/017_AB.svg" style="width:70%">
</div>


## Stream Ciphers

* Idea: stretch key + nonce into a keystream, then XOR it like a one-time pad
    + `keystream = PRG(key, nonce)` &nbsp;&middot;&nbsp; `c = m` $\oplus$ `keystream`
* The key is short, the keystream is not: *pseudorandomness* instead of true randomness
* **Never reuse a (key, nonce) pair**, that is a two-time pad
* Use: **ChaCha20**-Poly1305 (fast in software, no timing pitfalls)
<div style="text-align:center">
    <img src="./img/AB/s001_AB.svg" style="vertical-align:middle;width:60%">
</div>  

## Block Ciphers
<div style="display:flex; align-items:flex-start; gap:1em">

<div style="flex:6">
    
* A keyed **permutation** on fixed-size blocks 
* AES encrypts *exactly one block*, 128-bit blocks, 128/192/256-bit keys
* For real messages you need a **mode of operation**:
    + **ECB**: equal blocks &rarr; equal ciphertext. Leaks structure. **Never use**!
    + **CBC**: chaining + IV &nbsp;&middot;&nbsp; **CTR**: turns the block cipher into a stream cipher
* Modes, padding attacks and performance: *Cryptography*, *Security & Privacy Engineering*

<div style="flex:4; display:flex; flex-direction:column; gap:0.5em">
    <img src="./img/AB/e001_AB.svg" style="width:20%">
</div>

</div> 

### ECB Mode
<div style="text-align:center">
    <img src="./img/AB/e002_AB.svg" style="vertical-align:middle;width:60%">
</div> 

### Passive Attack
<div style="text-align:center">
    <img src="./img/AB/e003_AB.svg" style="vertical-align:middle;width:100%">
</div> 

### Passive Attack
<div style="text-align:center">
    <img src="./img/AB/e004_AB.svg" style="vertical-align:middle;width:100%">
</div> 

### Passive Attack
<div style="text-align:center">
    <img src="./img/AB/e005_AB.svg" style="vertical-align:middle;width:100%">
</div> 

### Passive Attack
<div style="text-align:center">
    <img src="./img/AB/e006_AB.svg" style="vertical-align:middle;width:100%">
</div> 

### Modes of Operation
* Plaintext vs ECB vs CBC
<div style="text-align:center">
    <img src="./img/tux.png" style="vertical-align:middle;width:100%">
</div>    
* Use authenticated encryption modes, more details in *Security and Privacy Engineering*

### Active Attack
<div style="text-align:center">
    <img src="./img/AB/e007_AB.svg" style="vertical-align:middle;width:100%">
</div> 

### Active Attack
<div style="text-align:center">
    <img src="./img/AB/e008_AB.svg" style="vertical-align:middle;width:100%">
</div> 

### Active Attack
<div style="text-align:center">
    <img src="./img/AB/e009_AB.svg" style="vertical-align:middle;width:100%">
</div> 

## Confidentiality Is Not Enough &rarr; AEAD

* Encrypted &ne; unmodified: CBC bit-flipping, padding oracles, truncation, replay, ...
* **AEAD** = *Authenticated Encryption with Associated Data*: one primitive for both
    + ciphertext + tag + associated data
    + *associated data* (headers, routing info) is authenticated, not encrypted
* Use: **AES-GCM**, AES-OCB, AES-CCM, or **ChaCha20-Poly1305**
* Combining by hand? **Encrypt-then-MAC** 
    + Decryption **verifies first**: reject before you touch a single plaintext byte

# Cryptographic Hash Functions & MACs

## Cryptographic Hash Functions

* Keyless, deterministic, fast, *fixed-size output*
    + $H: \{0,1\}^{*} \rightarrow \{0,1\}^n$ 
* Cryptographic hash function is "one-way", thus has to ensure 3 properties:
  1. **Preimage resistance**
  2. **Second preimage resistance**
  3. **Collision resistance**

### Cryptographic Hash Functions

**Preimage resistance**
<div style="background-color:#e7f1ff; color:#052c65; padding:10px 14px; border-left:5px solid #0d6efd; border-radius:4px">
Given a hash $h$ it should be infeasible to find <em>any</em> message $m$ (also called preimage) such that $h = H(m)$
</div>

Does this infeasibility depend on something elese besides the hash function?

**Hiding**: A hash function $H$ is hiding when a secret random value $r$ is chosen and used as additional input with $m$ such that $h = H(m||r)$, then also messages $m$ chosen from a small set are hidden. 

### Cryptographic Hash Functions

**Second preimage resistance**
<div style="background-color:#e7f1ff; color:#052c65; padding:10px 14px; border-left:5px solid #0d6efd; border-radius:4px">
Given a message $m$ it should be infeasible to find another message $m'$ such that $m \neq m'$ and $H(m) = H(m')$
</div>

In [8]:
# why changing the original input a little does not work
h = lambda m: hashlib.sha256(m).hexdigest()
print(h(b"crypto is fun")[:28], "...")
print(h(b"crypto is fun.")[:28], "...")                 # one byte more
print(bin(int(h(b"crypto is fun"), 16) ^
          int(h(b"crypto is fun."), 16)).count("1"), "of 256 bits flipped")

3efe42297d81750a84b8695528da ...
7eaa89a84b26a4f8dd480045bbb7 ...
128 of 256 bits flipped


### Cryptographic Hash Functions

**Collision resistance**
<div style="background-color:#e7f1ff; color:#052c65; padding:10px 14px; border-left:5px solid #0d6efd; border-radius:4px">
If should be infeasible to find <em>any</em> two messages $m \neq m'$ where $H(m) = H(m')$
</div>

### Collisions
* A cryptographic hash function maps one input from a large input space $\mathcal{I}$ to a smaller finite output space $\mathcal{O}$
<div style="text-align:center">
    <img src="./img/AB/h000_AB.svg" style="vertical-align:middle;width:70%">
</div> 

### Collisions
* A cryptographic hash function maps one input from a large input space $\mathcal{I}$ to a smaller finite output space $\mathcal{O}$
<div style="text-align:center">
    <img src="./img/AB/h001_AB.svg" style="vertical-align:middle;width:70%">
</div> 

### Collisions
* A cryptographic hash function maps one input from a large input space $\mathcal{I}$ to a smaller finite output space $\mathcal{O}$
<div style="text-align:center">
    <img src="./img/AB/h002_AB.svg" style="vertical-align:middle;width:70%">
</div> 

### Collisions
* A cryptographic hash function maps one input from a large input space $\mathcal{I}$ to a smaller finite output space $\mathcal{O}$
* A Collision
<div style="text-align:center">
    <img src="./img/AB/h003_AB.svg" style="vertical-align:middle;width:70%">
</div> 

### Collisions
* A cryptographic hash function maps one input from a large input space $\mathcal{I}$ to a smaller finite output space $\mathcal{O}$
* A Collision
* When the output space is $|\mathcal{O}|=2^{256}$, how many tries does an attacker need in the *worst case* to certainly find a collision?
<div style="text-align:center">
    <img src="./img/AB/h003_AB.svg" style="vertical-align:middle;width:70%">
</div> 

### Collisions
* A cryptographic hash function maps one input from a large input space $\mathcal{I}$ to a smaller finite output space $\mathcal{O}$
* When the output space is $|\mathcal{O}|=2^{256}$, an attacker needs $2^{256}+1$ tries in the *worst case* to certainly find a collision
<div style="text-align:center">
    <img src="./img/AB/h004_AB.svg" style="vertical-align:middle;width:70%">
</div> 

### Collisions
* Birthday bound: collisions after ~$2^{n/2}$ tries, SHA-256 = 128-bit collision security.

### Cryptographic Hash Functions
* Everywhere: Integrity, commitments, Merkle trees, `git`, blockchains, HMAC, KDFs, signatures, proof-of-work, bloom filter,  ...
* Use: **SHA-3**, **SHA-2**, BLAKE2/3

## Other Variants 

* **Passwords**: Never a bare SHA-256. Use **Argon2**, scrypt or bcrypt
    + deliberately slow and memory-hard
    + per-user **salt** $H(salt || m)$ &rarr; no rainbow tables 
* **Key derivation** from a shared secret: **HKDF**


## Message Authentication Code (MAC)

* `tag = MAC(k, m)`; the verifier recomputes it &rarr; integrity **and** authenticity
* Use **HMAC** (e.g. HMAC-SHA256), **Poly1305**, KMAC, CMAC
* Compare tags in **constant time** (`hmac.compare_digest`)
  + `==` leaks via timing side channel
* MAC &ne; signature: both sides share the key, so either could have made the tag
  &rarr; **no non-repudiation**

In [6]:
tag = hmac.new(b"secret", b"send 100€ to Bob", hashlib.sha256).hexdigest()
print(tag[:32], "...")
print(hmac.compare_digest(tag, tag)) # constant-time comparison

ce5889ebcba257313d15ba4123c26f49 ...
True


# Randomness

## Get Randomness
* `os.urandom()`, `getrandom()`, `/dev/urandom`
    + never `random.random()`
    + never use the current time as seed
* Randomness test tool [dieharder](https://rurban.github.io/dieharder/manual/dieharder.html)

# Asymmetric Cryptography

### The Problem Symmetric Crypto Cannot Solve

<div style="display:flex; align-items:flex-start; gap:1em">
<div style="flex:6">
    
* $n$ parties who all want to talk pairwise need $n(n-1)/2$ pre-shared keys for symmetric crypto
* 1000 parties $\approx$ 500 000 keys on each device 
* How do you set up a secure channel with a web-shop owner you have never met?

</div>

<div style="flex:4; display:flex; flex-direction:column; gap:0.5em">
    <img src="./img/AB/SymKeys.svg" style="width:50%">
</div>

</div>

In [3]:
n=1000
n*(n-1)/2

499500.0

In [4]:
n=5
n*(n-1)/2

10.0

# EOF